# 04. 검증: 모델이 '정착 결정'을 예측하는가, '사는 지역'을 맞히는가?

03번 모델은 정확도 0.98, ROC-AUC 0.996으로 매우 높았습니다. 그런데 타깃이 **현재 거주 시도가 수도권인지 여부**이고, SHAP 상위 변수에 `최종가중치`, `동부/읍면부 구분`, `직전 거주지(시군구)`가 있었습니다. 이 변수들은 응답자가 어느 지역에 사는지를 거의 직접 알려줍니다.

그래서 다음 세 가지를 비교해 성능이 어디서 오는지 확인합니다.
1. 03번과 같은 설정 (전체 변수)
2. 표본 가중치(`최종가중치`)만 제외
3. 지역을 직접 드러내는 변수를 모두 제외

비교를 단순하게 하기 위해 SMOTE 없이 같은 80/20 분할, 같은 LightGBM 설정을 사용합니다.

In [1]:
import re
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

df = pd.read_csv("2023_일반가구_20250526_59760.csv", encoding="cp949", low_memory=False)
df["비수도권_정착"] = (~df["시도"].isin([11, 28, 41])).astype(int)
print(df.shape)
print("비수도권 비율:", round(df["비수도권_정착"].mean(), 4))

(61260, 362)
비수도권 비율: 0.8754


비수도권 가구가 87.5%라서, 모두 '비수도권'으로 예측해도 정확도는 0.875입니다. 이 값이 비교 기준선입니다.

In [2]:
# 지역을 직접 드러내는 변수: 표본 가중치, 동부/읍면부, 직전 거주지, 이사 희망 지역 등
region_cols = [c for c in df.columns
               if re.search(r"시도|시군구|읍면동|지역|가중치|동부/읍면부|어디", c) and c != "시도"]
region_cols

['동부/읍면부 구분',
 '문6. 귀 댁의 주택 내 위치는 어디에 해당됩니까?',
 '문7. 귀 댁의 점유형태는 어디에 해당됩니까?',
 '문24. 현재주택 주거환경 만족도_3) 공공기관 접근용이성(시청/읍면동사무소/경찰서 등)',
 '문27. 현재주택으로 이사오기 바로 전에 어디서 사셨습니까?_시도',
 '문27. 현재주택으로 이사오기 바로 전에 어디서 사셨습니까?_시군구/읍면동',
 '문29. 귀 댁이 현재주택으로 이사오기 바로 전 주택의 점유형태는 어디에 해당됩니까?',
 '문30. 귀 댁이 현재주택으로 이사오기 바로 전 주택의 주택 내 위치는 어디에 해당됩니까?',
 '문35. 귀 댁은 앞으로 어느 지역으로 이사할 계획입니까?_시도',
 '문35. 귀 댁은 앞으로 어느 지역으로 이사할 계획입니까?_시군구/읍면동',
 '배문1. 가구주께서는 학교를 어디까지 마치셨습니까? 최종 졸업 학력 기준으로 응답해 주십시오.',
 '배문2-3-1. 다음 중 어디에 해당하십니까?(풀타임/파트타임)',
 '배문2-6. 가구주께서는 다음 중 어디에 해당하십니까?',
 '최종가중치']

키워드로 찾았기 때문에 `점유형태는 어디에 해당됩니까`처럼 지역과 무관한 문항도 일부 포함됩니다. 그래서 3번 결과는 보수적으로(성능이 낮게) 나올 수 있습니다.

In [3]:
def run(drop, label):
    X = df.drop(columns=["비수도권_정착", "시도"] + drop)
    num = X.select_dtypes(include=["int64", "float64"]).columns
    cat = X.select_dtypes(include=["object"]).columns
    X[num] = X[num].fillna(0)
    X[cat] = X[cat].fillna("없음")
    X = pd.get_dummies(X, drop_first=True)
    X.columns = [re.sub(r'[\[\]<>,:"{}]', "_", c) for c in X.columns]

    X_tr, X_te, y_tr, y_te = train_test_split(X, df["비수도권_정착"], test_size=0.2, random_state=42)
    model = lgb.LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.05,
                               random_state=42, verbose=-1).fit(X_tr, y_tr)
    proba = model.predict_proba(X_te)[:, 1]
    top = (pd.Series(model.booster_.feature_importance("gain"), index=X.columns)
             .sort_values(ascending=False).head(5).index.tolist())
    return {"설정": label, "변수 수": X.shape[1],
            "Accuracy": round(accuracy_score(y_te, proba > 0.5), 3),
            "ROC-AUC": round(roc_auc_score(y_te, proba), 3),
            "상위 변수 (gain)": top}

results = [
    run([], "1. 전체 변수 (03번과 동일)"),
    run(["최종가중치"], "2. 최종가중치 제외"),
    run(region_cols, "3. 지역 식별 변수 모두 제외"),
]
res = pd.DataFrame(results)
res[["설정", "변수 수", "Accuracy", "ROC-AUC"]]

,설정,변수 수,Accuracy,ROC-AUC
0,1. 전체 변수 (03번과 동일),446,0.979,0.995
1,2. 최종가중치 제외,445,0.972,0.993
2,3. 지역 식별 변수 모두 제외,432,0.957,0.983


In [4]:
for r in results:
    print(r["설정"])
    for f in r["상위 변수 (gain)"]:
        print("   -", f[:70])

1. 전체 변수 (03번과 동일)
   - 문27. 현재주택으로 이사오기 바로 전에 어디서 사셨습니까?_시군구/읍면동
   - 최종가중치
   - 동부/읍면부 구분
   - 문12. 현재 살고 계신 주택의 가격은 얼마입니까? - 현재 주택가격(만원)
   - 문4. 귀 댁의 주택 유형은 무엇입니까?
2. 최종가중치 제외
   - 문27. 현재주택으로 이사오기 바로 전에 어디서 사셨습니까?_시군구/읍면동
   - 문12. 현재 살고 계신 주택의 가격은 얼마입니까? - 현재 주택가격(만원)
   - 동부/읍면부 구분
   - 문21. 귀 댁이 거주하시는 곳의 주택면적은 얼마입니까?_1)주택사용면적(전용면적)_평
   - 문15. 현재 살고 계신 주택의 임차료는 얼마입니까?_보증금(만원)
3. 지역 식별 변수 모두 제외
   - 문12. 현재 살고 계신 주택의 가격은 얼마입니까? - 현재 주택가격(만원)
   - 문4. 귀 댁의 주택 유형은 무엇입니까?
   - 문15. 현재 살고 계신 주택의 임차료는 얼마입니까?_보증금(만원)
   - 문21. 귀 댁이 거주하시는 곳의 주택면적은 얼마입니까?_1)주택사용면적(전용면적)_평
   - 문19. 현재 살고 계신 주택은 언제 지어졌습니까?


In [5]:
res[["설정", "변수 수", "Accuracy", "ROC-AUC"]].to_csv("validation_results.csv", index=False, encoding="utf-8-sig")

## 해석

- 지역 식별 변수를 모두 빼도 정확도 0.957, AUC 0.983으로 여전히 높습니다.
- 하지만 이때 상위 변수는 **주택가격, 주택 유형, 보증금, 주택면적**입니다. 수도권과 비수도권의 집값 차이를 학습한 것으로 보입니다.
- 따라서 이 모델은 "어떤 가구가 비수도권에 **살고 있는가**"를 잘 구분하는 모델이며, 비수도권 정착을 **결정하는** 요인을 인과적으로 보여준다고 해석하기는 어렵습니다.
- 정착 결정 요인을 보려면 타깃을 "비수도권 거주 가구 중 수도권 이주 계획 여부"처럼 **의사결정**으로 다시 정의하고, 현재 주택가격처럼 거주 지역의 결과인 변수는 빼야 합니다.